# CI/CD Assignment
This notebook documents all 7 CI/CD tasks: CI pipeline setup, Jenkins, unit testing, integration
testing, performance testing, security testing, and continuous deployment (CD).

Each task section contains:
- Objective & tools
- Step-by-step instructions
- Ready-to-use config/code files (written to disk by the code cells)

> Run the code cells to generate the actual project files (sample app, workflow files,
> Jenkinsfile, test files, etc.) in a `project/` folder next to this notebook.

## Task 1 — Set Up a CI Pipeline (GitHub Actions)
**Objective:** Build a Node.js app automatically whenever new code is pushed.

**Steps:**
1. Create a simple Node.js app.
2. Push it to a GitHub repository.
3. Add a GitHub Actions workflow that installs deps and runs the build/tests on every push.
4. Push a commit and watch the workflow run in the *Actions* tab.

In [1]:
import os
os.makedirs('project', exist_ok=True)

# Simple Node.js app
with open('project/index.js', 'w') as f:
    f.write('''function add(a, b) { return a + b; }
console.log("App running:", add(2, 3));
module.exports = { add };
''')

with open('project/package.json', 'w') as f:
    f.write('''{
  "name": "ci-demo-app",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "start": "node index.js",
    "test": "jest"
  },
  "devDependencies": {
    "jest": "^29.0.0"
  }
}
''')
print("Created project/index.js and project/package.json")


Created project/index.js and project/package.json


In [2]:
import os
os.makedirs('project/.github/workflows', exist_ok=True)
with open('project/.github/workflows/ci.yml', 'w') as f:
    f.write('''name: CI Pipeline

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        node-version: [18.x]
    steps:
      - uses: actions/checkout@v4
      - name: Use Node.js ${{ matrix.node-version }}
        uses: actions/setup-node@v4
        with:
          node-version: ${{ matrix.node-version }}
      - run: npm install
      - run: npm test
      - run: npm run start
''')
print("Created project/.github/workflows/ci.yml")


Created project/.github/workflows/ci.yml


---
## Task 2 — Automated Build with Jenkins
**Objective:** Install Jenkins and configure a job that clones the repo and builds it automatically on push.

**Steps:**
1. Install Jenkins (local Docker container or cloud VM).
2. Create a Freestyle or Pipeline job.
3. Point it at the Git repository.
4. Add a build step (`npm install && npm test` or `mvn clean install`).
5. Configure a GitHub webhook / poll SCM trigger so it runs on every push.

In [3]:
# Run Jenkins locally via Docker (uncomment to execute where Docker is available)
jenkins_docker_cmd = "docker run -d -p 8080:8080 -p 50000:50000 --name jenkins jenkins/jenkins:lts"
print(jenkins_docker_cmd)


docker run -d -p 8080:8080 -p 50000:50000 --name jenkins jenkins/jenkins:lts


In [4]:
import os
os.makedirs('project', exist_ok=True)
with open('project/Jenkinsfile', 'w') as f:
    f.write('''pipeline {
    agent any

    triggers {
        pollSCM('* * * * *')   // or use a GitHub webhook instead of polling
    }

    stages {
        stage('Clone') {
            steps {
                git branch: 'main', url: 'https://github.com/<your-username>/<your-repo>.git'
            }
        }
        stage('Install') {
            steps {
                sh 'npm install'
            }
        }
        stage('Build') {
            steps {
                sh 'npm run start'
            }
        }
        stage('Test') {
            steps {
                sh 'npm test'
            }
        }
    }

    post {
        success { echo 'Build succeeded!' }
        failure { echo 'Build failed!' }
    }
}
''')
print("Created project/Jenkinsfile")


Created project/Jenkinsfile


---
## Task 3 — Write Unit Tests for an Application
**Objective:** Write unit tests for a small app (a calculator/to-do list) and run them inside the CI pipeline.

**Tooling:** Jest for the Node.js app created in Task 1.

In [5]:
import os
os.makedirs('project/__tests__', exist_ok=True)

# Small "calculator" style app with add/delete/update-like operations (a to-do list)
with open('project/todo.js', 'w') as f:
    f.write('''class TodoList {
  constructor() { this.items = []; }
  add(item) { this.items.push(item); return this.items; }
  delete(index) { this.items.splice(index, 1); return this.items; }
  update(index, newItem) { this.items[index] = newItem; return this.items; }
  list() { return this.items; }
}
module.exports = TodoList;
''')

with open('project/__tests__/todo.test.js', 'w') as f:
    f.write('''const TodoList = require('../todo');

test('adds an item', () => {
  const t = new TodoList();
  t.add('Buy milk');
  expect(t.list()).toEqual(['Buy milk']);
});

test('deletes an item', () => {
  const t = new TodoList();
  t.add('Buy milk');
  t.delete(0);
  expect(t.list()).toEqual([]);
});

test('updates an item', () => {
  const t = new TodoList();
  t.add('Buy milk');
  t.update(0, 'Buy oat milk');
  expect(t.list()).toEqual(['Buy oat milk']);
});
''')
print("Created project/todo.js and project/__tests__/todo.test.js")
print("Run locally with: cd project && npm install && npm test")


Created project/todo.js and project/__tests__/todo.test.js
Run locally with: cd project && npm install && npm test


These tests are already wired into the `ci.yml` (Task 1) and `Jenkinsfile` (Task 2) via `npm test`, so every push runs them automatically.


---
## Task 4 — Set Up Integration Testing for an API
**Objective:** Write integration tests for a REST API and automate them in a CI pipeline.

**Tooling:** A small Flask REST API + Postman/Newman.

In [6]:
import os
os.makedirs('project/api', exist_ok=True)

with open('project/api/app.py', 'w') as f:
    f.write('''from flask import Flask, jsonify, request

app = Flask(__name__)
DB = {}  # in-memory "database"

@app.route("/items", methods=["POST"])
def create_item():
    data = request.get_json()
    item_id = len(DB) + 1
    DB[item_id] = data["name"]
    return jsonify({"id": item_id, "name": data["name"]}), 201

@app.route("/items/<int:item_id>", methods=["GET"])
def get_item(item_id):
    if item_id not in DB:
        return jsonify({"error": "not found"}), 404
    return jsonify({"id": item_id, "name": DB[item_id]})

if __name__ == "__main__":
    app.run(debug=True)
''')

with open('project/api/requirements.txt', 'w') as f:
    f.write("flask\n")

# Postman collection (importable JSON) that exercises the API end-to-end
with open('project/api/postman_collection.json', 'w') as f:
    f.write('''{
  "info": { "name": "API Integration Tests", "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json" },
  "item": [
    {
      "name": "Create item",
      "request": {
        "method": "POST",
        "header": [{"key": "Content-Type", "value": "application/json"}],
        "body": {"mode": "raw", "raw": "{\"name\": \"Test Item\"}"},
        "url": {"raw": "http://localhost:5000/items", "host": ["localhost"], "port": "5000", "path": ["items"]}
      },
      "event": [{"listen": "test", "script": {"exec": ["pm.test('status is 201', () => pm.response.to.have.status(201));", "pm.environment.set('itemId', pm.response.json().id);"]}}]
    },
    {
      "name": "Get item",
      "request": {
        "method": "GET",
        "url": {"raw": "http://localhost:5000/items/{{itemId}}", "host": ["localhost"], "port": "5000", "path": ["items", "{{itemId}}"]}
      },
      "event": [{"listen": "test", "script": {"exec": ["pm.test('status is 200', () => pm.response.to.have.status(200));"]}}]
    }
  ]
}
''')
print("Created project/api/app.py, requirements.txt, postman_collection.json")


Created project/api/app.py, requirements.txt, postman_collection.json


In [7]:
ci_snippet = '''
  integration-tests:
    needs: build
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Start Flask API
        run: |
          pip install -r project/api/requirements.txt
          nohup python project/api/app.py &
          sleep 3
      - name: Run Newman (Postman CLI)
        run: |
          npm install -g newman
          newman run project/api/postman_collection.json
'''
print(ci_snippet)
print("Add this job to project/.github/workflows/ci.yml (or a Jenkins stage using 'newman run ...').")



  integration-tests:
    needs: build
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Start Flask API
        run: |
          pip install -r project/api/requirements.txt
          nohup python project/api/app.py &
          sleep 3
      - name: Run Newman (Postman CLI)
        run: |
          npm install -g newman
          newman run project/api/postman_collection.json

Add this job to project/.github/workflows/ci.yml (or a Jenkins stage using 'newman run ...').


---
## Task 5 — Automate Performance Testing in a CI/CD Pipeline
**Objective:** Load-test the API using k6 (or JMeter) and run it automatically after every build.

**Tooling:** k6 (simpler to script and run in CI than JMeter).

In [8]:
import os
os.makedirs('project/perf', exist_ok=True)
with open('project/perf/load_test.js', 'w') as f:
    f.write('''import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  vus: 20,          // 20 virtual users
  duration: '30s',
};

export default function () {
  const res = http.get('http://localhost:5000/items/1');
  check(res, { 'status is 200 or 404': (r) => [200, 404].includes(r.status) });
  sleep(1);
}
''')
print("Created project/perf/load_test.js")
print("Run with: k6 run project/perf/load_test.js")
print("CI step example: 'docker run --network=host -v $PWD/project/perf:/scripts grafana/k6 run /scripts/load_test.js'")


Created project/perf/load_test.js
Run with: k6 run project/perf/load_test.js
CI step example: 'docker run --network=host -v $PWD/project/perf:/scripts grafana/k6 run /scripts/load_test.js'


---
## Task 6 — Implement Basic Security Testing in a CI Pipeline
**Objective:** Scan the app for vulnerabilities with OWASP ZAP (dynamic) and/or SonarQube (static) on every push.

In [9]:
zap_ci_step = '''
  security-scan:
    needs: integration-tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Start app
        run: |
          pip install -r project/api/requirements.txt
          nohup python project/api/app.py &
          sleep 3
      - name: OWASP ZAP Baseline Scan
        uses: zaproxy/action-baseline@v0.12.0
        with:
          target: 'http://localhost:5000'
'''
sonar_ci_step = '''
      - name: SonarQube Scan
        uses: SonarSource/sonarqube-scan-action@v2
        env:
          SONAR_TOKEN: ${{ secrets.SONAR_TOKEN }}
'''
print(zap_ci_step)
print(sonar_ci_step)



  security-scan:
    needs: integration-tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Start app
        run: |
          pip install -r project/api/requirements.txt
          nohup python project/api/app.py &
          sleep 3
      - name: OWASP ZAP Baseline Scan
        uses: zaproxy/action-baseline@v0.12.0
        with:
          target: 'http://localhost:5000'


      - name: SonarQube Scan
        uses: SonarSource/sonarqube-scan-action@v2
        env:
          SONAR_TOKEN: ${{ secrets.SONAR_TOKEN }}



---
## Task 7 — Automate Deployment to a Staging Environment (CD)
**Objective:** After a successful build + tests, automatically deploy the app to staging using Docker/Kubernetes.

In [10]:
import os
os.makedirs('project/api', exist_ok=True)
with open('project/Dockerfile', 'w') as f:
    f.write('''FROM python:3.11-slim
WORKDIR /app
COPY api/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY api/ .
EXPOSE 5000
CMD ["python", "app.py"]
''')

with open('project/k8s-deployment.yaml', 'w') as f:
    f.write('''apiVersion: apps/v1
kind: Deployment
metadata:
  name: ci-demo-api-staging
spec:
  replicas: 2
  selector:
    matchLabels:
      app: ci-demo-api
  template:
    metadata:
      labels:
        app: ci-demo-api
    spec:
      containers:
        - name: ci-demo-api
          image: <your-dockerhub-username>/ci-demo-api:latest
          ports:
            - containerPort: 5000
---
apiVersion: v1
kind: Service
metadata:
  name: ci-demo-api-staging-svc
spec:
  type: NodePort
  selector:
    app: ci-demo-api
  ports:
    - port: 5000
      targetPort: 5000
''')

deploy_ci_step = '''
  deploy-staging:
    needs: security-scan
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build & push image
        run: |
          docker build -t <your-dockerhub-username>/ci-demo-api:latest -f project/Dockerfile project
          echo "${{ secrets.DOCKERHUB_TOKEN }}" | docker login -u <your-dockerhub-username> --password-stdin
          docker push <your-dockerhub-username>/ci-demo-api:latest
      - name: Deploy to staging cluster
        run: |
          kubectl apply -f project/k8s-deployment.yaml
'''
print("Created project/Dockerfile and project/k8s-deployment.yaml")
print(deploy_ci_step)


Created project/Dockerfile and project/k8s-deployment.yaml

  deploy-staging:
    needs: security-scan
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build & push image
        run: |
          docker build -t <your-dockerhub-username>/ci-demo-api:latest -f project/Dockerfile project
          echo "${{ secrets.DOCKERHUB_TOKEN }}" | docker login -u <your-dockerhub-username> --password-stdin
          docker push <your-dockerhub-username>/ci-demo-api:latest
      - name: Deploy to staging cluster
        run: |
          kubectl apply -f project/k8s-deployment.yaml



---
## Repository Structure Summary
```
project/
├── .github/workflows/ci.yml
├── Jenkinsfile
├── index.js / package.json
├── todo.js
├── __tests__/todo.test.js
├── api/ (app.py, requirements.txt, postman_collection.json)
├── perf/load_test.js
├── Dockerfile
└── k8s-deployment.yaml
```